# FieldAid Walkthrough Demo## Gemma 4 Good HackathonThis notebook clones the full repository (including walkthrough media), sets up Ollama with Gemma 4 E4B, starts the server, and launches the walkthrough demo.

### Step 1: Clone Repository and Install

In [ ]:
!git clone --branch wt/walkthrough --depth 1 https://github.com/Kushalk0677/fieldaid-offline-disaster-copilot.git%cd fieldaid-offline-disaster-copilotprint("Cloned repo with walkthrough folder.")

In [ ]:
!pip install -r requirements.txt -qimport sysprint(f"Python {sys.version}")from app.main import appprint("FastAPI app loaded successfully.")

### Step 2: Install Ollama and Pull Gemma 4 E4B (~9.6 GB)Leave this running. Takes 5-10 minutes.

In [ ]:
!apt-get update -qq && apt-get install -y -qq zstd curl > /dev/null 2>&1!curl -fsSL https://ollama.com/install.sh | shimport os, subprocess, timeos.system("killall -9 ollama 2>/dev/null || true")print("Starting Ollama server...")subprocess.Popen(["ollama", "serve"], stdout=subprocess.PIPE, stderr=subprocess.PIPE)time.sleep(3)print("Pulling gemma4:e4b (~9.6 GB). Stand by...")!ollama pull gemma4:e4bprint("\nVerification:")!ollama run gemma4:e4b "Say hello in one word." --quiet

### Step 3: Configure and Start Server

In [ ]:
import osfrom pathlib import Pathos.environ["FIELDAID_RUNTIME_CACHE"] = str(Path.home() / ".fieldaid_runtime")os.environ["PYTHONPATH"] = "."Path("data/uploads").mkdir(parents=True, exist_ok=True)Path(os.environ["FIELDAID_RUNTIME_CACHE"]).mkdir(parents=True, exist_ok=True)print("Runtime configured.")classifier = Path("models/fieldaid-medic-image-classifier-final/best.pt")yolo = Path("yolo11n.pt")for name, p in [("Classifier", classifier), ("YOLO", yolo)]:    if p.exists():        print(f"  {name}: found ({p.stat().st_size / 1e6:.1f} MB)")    else:        print(f"  {name}: missing")wt_media = Path("walkthrough/media")if wt_media.exists():    files = [f.name for f in wt_media.iterdir()]    print(f"  Walkthrough media: {len(files)} files")else:    print("  Walkthrough media: not found")

In [ ]:
import threading, timedef run_server():    os.system("uvicorn app.main:app --host 0.0.0.0 --port 8000 > /tmp/fieldaid.log 2>&1")print("Starting FieldAid server on port 8000...")server_thread = threading.Thread(target=run_server, daemon=True)server_thread.start()time.sleep(4)import httpxtry:    resp = httpx.get("http://localhost:8000", timeout=5)    print(f"Server running (status {resp.status_code})")    print("\nClick the preview panel above (or visit http://localhost:8000)")    print("Then click \x1b[32mStart Walkthrough\x1b[0m button in the bottom-right corner.")except Exception as e:    print(f"Server not ready yet: {e} (try this cell again)")

### Step 4: Walkthrough Instructions1. Click the **preview panel** above (Colab will open it)2. On the webpage, click **Start Walkthrough** (bottom-right corner, pulses green)3. Click **Next** to advance through each step   - Forms auto-fill with scenario data   - Images and videos auto-load from walkthrough media   - Gemma 4 E4B runs locally for each analysis4. Use **Previous** to review past steps5. Click **End** (top-right X) when finishedThe walkthrough runs 11 steps covering shelter intake, damage assessment, video scanning, trust gates, dashboard, incidents, Discord handoff, and sync export.